In [ ]:
import pandas as pd
import gzip
import json

In [ ]:
INPUT_TELEGRAM_DATA_PATH = "../data/youtube_links.jsonl"
INPUT_TEXT_PATH = "../data/preprocessed_english_titles.csv"
INPUT_YT_DATA_PATH = "../youtube_titulos_saida.csv"

OUTPUT_PATH = "../data/youtube_telegram_cross.csv"

In [ ]:
df_titles = pd.read_csv(INPUT_TEXT_PATH)
df_infos = pd.read_csv(INPUT_YT_DATA_PATH)
df_telegram = pd.read_json(INPUT_TELEGRAM_DATA_PATH, lines=True)

print(f"Preprocessed titles/descriptions: {len(df_titles)} entries")
display(df_titles.head())
print(f"Video metrics/metadata: {len(df_infos)} entries")
display(df_infos.head())
print(f"Telegram message occurrences: {len(df_telegram)} entries")
display(df_telegram.head())

Preprocessed titles/descriptions: 686626 entries


,Unnamed: 0,video_id,title,description,title_description,lang,clean_text
0,0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Free bus travel for migrants scrapped. For 5 m...,en,free travel migrant scrap minute
1,1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",What is Spiritual Warfare? This charge I commi...,en,spiritual warfare charge commit unto thee timo...
2,2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...","80 Putins Have Layers In February 2024, Tucker...",en,putin layer february tucker carlson stand onio...
3,3,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,en,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...
4,4,AKU0RokegSo,Los Angeles rain: Studio City homes evacuated ...,An atmospheric river storm has caused mudslide...,Los Angeles rain: Studio City homes evacuated ...,en,angeles rain studio city home evacuate mudslid...


Video metrics/metadata: 1183816 entries


,video_id,title,description,channel_title,published_at,youtube_video_link,view_count,like_count,comment_count
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Joe Marsh,2024-04-10T15:05:37Z,https://www.youtube.com/watch?v=jEKzQV5oajY,1057.0,152.0,32.0
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",WWURD,2023-10-18T14:39:18Z,https://www.youtube.com/watch?v=xrGGce8cmx8,65.0,6.0,1.0
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...",The Mosaic Ark,2024-02-22T07:30:05Z,https://www.youtube.com/watch?v=uaozGpSc4nc,330.0,8.0,6.0
3,TwACgO_oPq4,Anacondaz — Акуле плевать (Official Music Video),#Anacondaz #Акулеплевать #Безпаники\n\nLong St...,ANACONDAZ,2014-05-20T09:29:45Z,https://www.youtube.com/watch?v=TwACgO_oPq4,2321870.0,24565.0,561.0
4,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,https://www.youtube.com/watch?v=A59ftbhsQUE,806.0,136.0,18.0


Telegram message occurrences: 1920792 entries


,url,occurrences,id
0,http://www.youtube.com/watch?v=0BskDtKkX6Q,"[{'id': 12771, 'folder': 'channel_1205583420',...",0BskDtKkX6Q
1,http://www.youtube.com/watch?v=1L2V9Ml9Hi0,"[{'id': 88757, 'folder': 'channel_1181961026',...",1L2V9Ml9Hi0
2,http://www.youtube.com/watch?v=2EMyoELeAC0,"[{'id': 744, 'folder': 'channel_1639057388', '...",2EMyoELeAC0
3,http://www.youtube.com/watch?v=6PmrsXPrP0I,"[{'id': 274202, 'folder': 'channel_1153527121'...",6PmrsXPrP0I
4,http://www.youtube.com/watch?v=OKN8dFO_ZLA,"[{'id': 533712, 'folder': 'channel_1828686465'...",OKN8dFO_ZLA


### **1. Extracting the IDs from the links**

In [2]:
import re

def get_ID(url:str) -> str:
    #match = re.search(
    #        r"(?:v=|youtu\.be/|/v/|/embed/|/live/|/shorts/)([a-zA-Z0-9_-]{11})",
    #        url
    #    )
    match = re.search(
                r'(?:v=|\/|be\/|embed\/|shorts\/|watch\?v=)([a-zA-Z0-9_-]{11})',
                url
            )
    if match:
        return match.group(1)
    return None

In [ ]:
objects = []
with open(INPUT_TELEGRAM_DATA_PATH, "rt") as f:
    for line in f:
        obj = json.loads(line)
        obj["id"] = get_ID(obj["url"])
        objects.append(obj)


In [ ]:
with open(INPUT_TELEGRAM_DATA_PATH, "wt") as f:
    for obj in objects:
        f.write(json.dumps(obj) + "\n")

___
### **2. Combining the interesting columns**

#### *2.1 Combining with df_titles*

In [2]:
df_final = pd.DataFrame({
    "video_id": df_titles["video_id"],
    "text": df_titles["clean_text"]
})

#### *2.2 Combining with df_infos*

In [3]:
df_final = df_final.merge(
    df_infos[["video_id", "channel_title", "published_at", "view_count", "like_count", "comment_count"]],
    on="video_id",
    how="left"
)

#### *2.3 Combining with df_telegram*
But this has duplicate IDs, because different links can lead to the same ID, so that needs to be fixed first.

In [4]:
counts = df_telegram["id"].value_counts()
duplicates = counts[counts > 1]
print(duplicates)

id
_YIe_4LRQfI    306
hBMoPUAeLnY    284
c5wnRebTKKY    271
9-zF54xbFw0    180
8sEsfotldIY    145
              ... 
pBDI7xv__Fo      2
1fsKvW8fcDw      2
p5attvG06uQ      2
KFppEuJRTO4      2
o821QX5LFpg      2
Name: count, Length: 235500, dtype: int64


In [5]:
video_id = "o821QX5LFpg"

# Filtra apenas as linhas com esse video_id
occurrences = df_telegram[df_telegram["id"] == video_id]

# Exibe todas as ocorrências
display(occurrences)
display(occurrences["occurrences"].tolist())

,url,occurrences,id
142450,https://youtu.be/o821QX5LFpg?si=DDetjzACmDqnrMra,"[{'id': 11707, 'folder': 'channel_1577438065',...",o821QX5LFpg
872570,https://youtu.be/o821QX5LFpg?si=bT00jO37-EOlT7ba,"[{'id': 10923, 'folder': 'channel_1385694192',...",o821QX5LFpg


[[{'id': 11707, 'folder': 'channel_1577438065', 'date': '2023-11-30'}],
 [{'id': 10923, 'folder': 'channel_1385694192', 'date': '2023-12-02'}]]

In [6]:
df_telegram = df_telegram.rename(columns={"id": "video_id"})

#Como um id pode ter mais de um link, agrupa essas ocorrências
df_telegram_unique = df_telegram.groupby("video_id")["occurrences"].apply(list).reset_index()

df_telegram_unique = (
    df_telegram.groupby("video_id")["occurrences"]
    .apply(lambda x: [item for sublist in x for item in sublist])
    .reset_index()
)

In [7]:
video_id = "o821QX5LFpg"

# Filtra apenas as linhas com esse video_id
occurrences = df_telegram_unique[df_telegram_unique["video_id"] == video_id]

# Exibe todas as ocorrências
display(occurrences)
display(occurrences["occurrences"].tolist())

,video_id,occurrences
1139383,o821QX5LFpg,"[{'id': 11707, 'folder': 'channel_1577438065',..."


[[{'id': 11707, 'folder': 'channel_1577438065', 'date': '2023-11-30'},
  {'id': 10923, 'folder': 'channel_1385694192', 'date': '2023-12-02'}]]

In [8]:
df_final = df_final.merge(
    df_telegram_unique,
    on="video_id",
    how="left"
)

In [9]:
display(df_final.head())

,video_id,text,channel_title,published_at,view_count,like_count,comment_count,occurrences
0,jEKzQV5oajY,free travel migrant scrap minute,Joe Marsh,2024-04-10T15:05:37Z,1057.0,152.0,32.0,"[{'id': 1416, 'folder': 'channel_1556142220', ..."
1,xrGGce8cmx8,spiritual warfare charge commit unto thee timo...,WWURD,2023-10-18T14:39:18Z,65.0,6.0,1.0,"[{'id': 60799, 'folder': 'channel_1466271872',..."
2,uaozGpSc4nc,putin layer february tucker carlson stand onio...,The Mosaic Ark,2024-02-22T07:30:05Z,330.0,8.0,6.0,"[{'id': 40486, 'folder': 'channel_1235978663',..."
3,A59ftbhsQUE,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,806.0,136.0,18.0,"[{'id': 449843, 'folder': 'channel_1571505334'..."
4,AKU0RokegSo,angeles rain studio city home evacuate mudslid...,FOX 11 Los Angeles,2024-02-06T01:45:29Z,60863.0,406.0,152.0,"[{'id': 104116, 'folder': 'channel_1438734111'..."


In [ ]:
df_final.to_csv(OUTPUT_PATH)